# Kata 22 — Configuración de MCP Servers

> Spec: [`specs/022-mcp-config/spec.md`](../../specs/022-mcp-config/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

Este kata es de **configuración**, no de runtime de la API. Lo demuestro con
parsing real de los archivos JSON y validación. La parte API (¿el modelo
escoge MCP tool vs built-in?) corre al final con un test reproducible.

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap

client, settings = bootstrap(budget_calls=4)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

`.mcp.json` define MCP servers para todo el equipo (versionado en git).
`~/.claude.json` define servers personales del dev. Las credenciales se
inyectan via `${ENV_VAR}` para no commitear secrets.

## 2. Por qué importa

Un token hardcoded en `.mcp.json` versionado se publica al primer push.
Un server de equipo en `~/.claude.json` no se replica al onboarding.

## 3. Configuración correcta — project + user scope

In [ ]:
import json, os, re, tempfile, pathlib

PROJECT_MCP = {
    "mcpServers": {
        "github": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-github"],
            "env": {"GITHUB_TOKEN": "${GITHUB_TOKEN}"}
        },
        "internal-docs": {
            "command": "node",
            "args": ["./scripts/mcp-docs.js"]
        }
    }
}

USER_MCP = {
    "mcpServers": {
        "experimental": {"command": "python3", "args": ["./local-mcp.py"]}
    }
}

def expand_env(value: str) -> str:
    return re.sub(r"\$\{([A-Z_][A-Z0-9_]*)\}", lambda m: os.environ.get(m.group(1), ""), value)

def resolve_servers(project: dict, user: dict) -> dict:
    merged = {}
    for scope, cfg in [("user", user), ("project", project)]:
        for name, server in cfg.get("mcpServers", {}).items():
            resolved = dict(server)
            resolved["env"] = {k: expand_env(v) for k, v in server.get("env", {}).items()}
            resolved["_scope"] = scope
            merged[name] = resolved
    return merged

# Simular carga: env var GITHUB_TOKEN puede estar set o no
os.environ.setdefault("GITHUB_TOKEN", "ghp_dummy_for_demo")
servers = resolve_servers(PROJECT_MCP, USER_MCP)
for name, cfg in servers.items():
    safe = dict(cfg)
    if "env" in safe:
        safe["env"] = {k: ("<redacted>" if "TOKEN" in k or "KEY" in k else v) for k, v in safe["env"].items()}
    print(f"{name} ({cfg['_scope']}): {json.dumps(safe, ensure_ascii=False)}")

### 3.1 Aserción defensiva — secrets no commiteados

In [ ]:
def assert_no_hardcoded_secrets(config: dict):
    bad = []
    for name, server in config.get("mcpServers", {}).items():
        for k, v in server.get("env", {}).items():
            if not isinstance(v, str): continue
            # Heurística: si k contiene TOKEN/KEY/SECRET y el valor NO es ${VAR}, es hardcoded
            if any(kw in k.upper() for kw in ("TOKEN","KEY","SECRET","PASSWORD")):
                if not v.startswith("${"):
                    bad.append(f"{name}.{k}: hardcoded value")
    return bad

# Project bien
print("project hardcoded?", assert_no_hardcoded_secrets(PROJECT_MCP) or "OK")

# Caso malo (a propósito)
BAD = {"mcpServers":{"github":{"command":"npx","args":[],"env":{"GITHUB_TOKEN":"ghp_realtokenLeakedToGit"}}}}
print("bad config detected:", assert_no_hardcoded_secrets(BAD))

## 4. Anti-patrón — el modelo prefiere MCP tool cuando un built-in resuelve

In [ ]:
# Tool MCP que duplica funcionalidad de Grep
SEARCH_FILES_TOOL = {
    "name": "search_files",
    "description": "Searches code in files (custom MCP tool).",
    "input_schema": {"type":"object","properties":{"pattern":{"type":"string"}},"required":["pattern"]},
}

# La descripción genérica + la tendencia a usar tools "custom" hace que
# el modelo prefiera search_files sobre Grep cuando ambos están disponibles.
# Lección de Kata 21: si vas a exponer un MCP tool, su descripción debe
# justificar cuándo SÍ usarlo (ej. busca en sistemas externos al filesystem).

resp = client.messages.create(
    system="Tienes search_files (MCP custom). Usa la herramienta apropiada para buscar contenido.",
    messages=[{"role":"user","content":"Busca todas las menciones de processRefund en el repo."}],
    tools=[SEARCH_FILES_TOOL],
    tool_choice={"type":"any"},
)
tu = next((b for b in resp.content if b.type == "tool_use"), None)
print(f"tool invocado: {tu.name if tu else '(none)'} con {tu.input if tu else ''}")
print("Lección: si tu MCP tool no diferencia su scope (filesystem vs sistema externo)")
print("contra los built-ins, el modelo lo va a usar por preferencia. Mejora la descripción")
print("o renombra: search_files_in_jira, search_files_in_confluence, etc.")

## 5. Argumento de certificación

- **Project scope** (`.mcp.json`) viaja con el repo y se descubre al
  conectar. **User scope** (`~/.claude.json`) es personal.
- **Env var expansion** es el patrón estándar para credenciales — nunca
  hardcoded.
- **MCP tools deben justificar su existencia** vs los built-ins; si no,
  son ruido.
- Conexión con Kata 21: la descripción del MCP tool debe ser explícita
  sobre cuándo usarlo en lugar de un built-in.

## 6. Auto-evaluación

**1. ¿Qué pasa si dos servers exponen un tool con el mismo nombre?**

Comportamiento undefined a nivel SDK; en práctica el último cargado gana
o uno de los dos no se registra. La defensa: namespace en el nombre del
tool (e.g., `github_create_issue` vs `jira_create_issue`).

**2. ¿Cómo se manejan los secrets en CI cuando `.mcp.json` se ejecuta allí?**

Los CI runners exponen secrets como env vars (GitHub Actions:
`secrets.GITHUB_TOKEN`). El `${GITHUB_TOKEN}` en `.mcp.json` resuelve
desde el runner, sin commit a git, sin exposure en logs.

**3. ¿Cuándo prefiero un MCP resource sobre un MCP tool?**

Cuando expongo un **catálogo de contenido** (issues, docs, schemas) que
el agente necesita ver pero no actuar sobre — los resources se inyectan
como contexto, evitando llamadas exploratorias innecesarias.